# Laboratoires CNRS grands gap

In [1]:
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import pygadm
from shapely.geometry import Point
from ipyleaflet import GeoJSON, Map, basemaps
import folium
import openpyxl as pyxl
user = 'scola'
import sys
sys.path.insert (0, f'/home/{user}/modules/routines/')
import routines as rt

path2sources = './sources/'
path2figures = './figures/'

In [2]:
def majuscules_aux_villes (nom):
    parts = nom.split ('-')
    majname = ''
    for part in parts:
        part = part.capitalize ()
        majname += part + '-'
    majname = majname [:-1]
    return majname.replace ('Sur', 'sur')

In [3]:
# Load cities geolocation
file = path2sources + 'villes.csv'
with open (file, 'r') as f:
    content = f.read ()
    f.close ()
lines = content.split ('\n')
villes = {}
for line in lines [:-1]:
    nom, lat, long = line.split (',')
    villes [majuscules_aux_villes (nom)] = (lat, long)

gdf_villes = gpd.GeoDataFrame(
    {"nom": list(villes.keys())},
    geometry=[Point(lon, lat) for lat, lon in villes.values()],
    crs="EPSG:4326",
)

In [4]:
# Load labs data in a dict
datafile = path2sources + 'Liste Labo Equipes Personnes_ml_yc.xlsx'
data = pyxl.load_workbook(filename = datafile, data_only = True)
# open the first sheet as var sheet
datasheet = data.worksheets[0]

Labs = {}

mainkey = 'UMR/UPR' 

# lecture des noms de colonnes
columns = {}
j = 1
colname = datasheet.cell (row = 1, column = j).value
while (colname != None ):
    columns [colname] = j
    j += 1
    colname = datasheet.cell (row = 1, column = j).value

i = 2
rowname = datasheet.cell (row = i, column = columns [mainkey]).value
while (rowname != None):
    lab = {}
    for column_name in columns.keys ():
        lab [column_name] = datasheet.cell (row = i, column = columns [column_name]).value
    lab ['Ville'] = majuscules_aux_villes (lab ['Ville'])
    lab ['Coordinates'] = villes [lab ['Ville']]
    Labs [lab [mainkey]] = lab
    i += 1
    rowname = datasheet.cell (row = i, column = columns [mainkey]).value

In [5]:
# Convert data dict to GeoPandasDataFrame 
cols = ['Acronyme labo', 'UMR/UPR', 'Institut principal', 'Ville']
gdf_Labs = gpd.GeoDataFrame(
    pd.DataFrame (Labs).T [cols],
    geometry=[Point(lon, lat) for lat, lon in [d ['Coordinates'] for d in Labs.values ()]],
    crs="EPSG:4326",
)

In [6]:
VersaillesLabs = gdf_Labs.loc[gdf_Labs['Ville'].isin(['Versailles'])]
VersaillesLabs

,Acronyme labo,UMR/UPR,Institut principal,Ville,geometry
UMR8635,GEMaC,UMR8635,Physique,Versailles,POINT (2.1301 48.8014)
UMR8180,ILV,UMR8180,Chimie,Versailles,POINT (2.1301 48.8014)
UMR8029,SATIE,UMR8029,Ingénierie,Versailles,POINT (2.1301 48.8014)


In [7]:
# Load other labs data in a dict
datafile = path2sources + 'Liste Labo Equipes Personnes_ml_yc_v2.xlsx'
data = pyxl.load_workbook(filename = datafile, data_only = True)
# open the first sheet as var sheet
datasheet = data.worksheets[3]

otherLabs = {}

mainkey = 'Acronyme labo' 

# lecture des noms de colonnes
columns = {}
j = 1
colname = datasheet.cell (row = 1, column = j).value
while (colname != None ):
    columns [colname] = j
    j += 1
    colname = datasheet.cell (row = 1, column = j).value

i = 2
rowname = datasheet.cell (row = i, column = columns [mainkey]).value
while (rowname != None):
    lab = {}
    for column_name in columns.keys ():
        lab [column_name] = datasheet.cell (row = i, column = columns [column_name]).value
    lab ['Ville'] = majuscules_aux_villes (lab ['Ville'])
    lab ['Coordinates'] = villes [lab ['Ville']]
    otherLabs [lab [mainkey]] = lab
    i += 1
    rowname = datasheet.cell (row = i, column = columns [mainkey]).value

In [8]:
# Convert data dict to GeoPandasDataFrame 
cols = ['Acronyme labo', 'Institut principal', 'Ville']
gdf_otherLabs = gpd.GeoDataFrame(
    pd.DataFrame (otherLabs).T [cols],
    geometry=[Point(lon, lat) for lat, lon in [d ['Coordinates'] for d in otherLabs.values ()]],
    crs="EPSG:4326",
)

In [10]:
# CNRS labs
category = 'Ville'
sites = list (gdf_Labs [category].unique ())
Labsonsites = {}

for site in sites:
    subset = gdf_Labs.loc [gdf_Labs [category].isin([site])]
    labsonsite = ''
    i = 0
    for labcode, lab in subset.iterrows ():
        labsonsite += f'{lab ['Acronyme labo']} {labcode} ({lab ['Institut principal']}), '
        i += 1
    d = {
        'ville': site,
        'nombre de labos CNRS': i,
        'noms des labos': labsonsite [:-2],
        'geometry': lab ['geometry']
    }
    Labsonsites [site] = d    
    
gdf_Labsonsites = gpd.GeoDataFrame (
    pd.DataFrame (Labsonsites).T,
    crs="EPSG:4326"
)


# other labs
category = 'Ville'
sites = list (gdf_otherLabs [category].unique ())
otherLabsonsites = {}

for site in sites:
    subset = gdf_otherLabs.loc [gdf_otherLabs [category].isin([site])]
    labsonsite = ''
    i = 0
    for labcode, lab in subset.iterrows ():
        labsonsite += f'{lab ['Acronyme labo']} ({lab ['Institut principal']}), '
        i += 1
    d = {
        'ville': site,
        'nombre de labos CNRS': i,
        'noms des labos': labsonsite [:-2],
        'geometry': lab ['geometry']
    }
    otherLabsonsites [site] = d  

gdf_otherLabsonsites = gpd.GeoDataFrame (
    pd.DataFrame (otherLabsonsites).T,
    crs="EPSG:4326"
)

In [13]:
# display on map
m = gdf_Labsonsites.explore (popup = True,
                         marker_kwds = {'radius': 5},
                         color = '#00294bff',
                         name = 'CNRS labs'
                         )



gdf_otherLabsonsites.explore (m = m, popup = True,
                         marker_kwds = {'radius': 5},
                         color = '#e30a1dff',
                         name = 'non CNRS labs'
                         )
folium.LayerControl().add_to(m)
m